# Orchestration Agent

> This agent is designed to call plan agent actions from a given prompt and use expert agents efficiently

In [ ]:
#| default_exp agents.orchestration_agent

In [ ]:
#| export
import datetime
from dataclasses import dataclass
from typing import Literal
from httpx import AsyncClient, ConnectError, ConnectTimeout, RemoteProtocolError
import requests

from pydantic import BaseModel, Field
from rich.prompt import Prompt

from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.messages import ModelMessage
from pydantic_ai.usage import Usage, UsageLimits

In [ ]:
#| export
class _OrchestrationPlan(BaseModel):
    """Plan for orchestration of agent models and tools to solve a users main goal"""
    main_goal: str
    sub_goals: list[str] = Field(description=(
        "List of sub goals needed to achieve the main goal, "
        "sub goals should be simple and achievable with the tools and models available. "
        "Sub goals can reference other sub goals or the main goal. "
        "Sub goals must include a reason for why they are needed to achieve the main goal.")
        )
    plan: list[str] =  Field(description="List of steps to achieve the main goal along with sub goals")

In [ ]:
#| hide
from duckduckgo_search import DDGS

In [ ]:
@dataclass
class Deps:
    ...

In [ ]:
@dataclass
class SearchSolution:
    sub_goal: str
    search_results: list[str]
    result_summary: str

In [ ]:
#| export
def build_orchestration_agent():
    return Agent[None, _OrchestrationPlan](
    "openai:gpt-4o-mini", 
    result_type=_OrchestrationPlan,
    system_prompt=(
        "You are an expert at orchestrating models and tools to solve complex problems. "
        "You break down problems into simple sub-goals and plan out the steps needed to achieve them. "
        "You have been asked to help a user achieve their main goal. "
        "The user has provided a prompt you must determine the main goal to best help them. "
        "You must then determine the sub-goals needed to achieve the main goal and plan out the steps needed to achieve them. "
        "You are able to use a number of tools and models to help you achieve the main goal. "
    )
    )

In [ ]:
orchestration_agent = build_orchestration_agent()

In [ ]:
@dataclass
class SearchDeps:
    search_client: DDGS

In [ ]:
search_agent = Agent[DDGS, SearchSolution|None](
    "openai:gpt-4o",
    result_type=SearchSolution,
    deps_type=SearchDeps,
    system_prompt=(
        "You are an expert at searching the web for information. "
        "You can quickly find information on any topic and provide a summary of the results. "
        "You have been asked to help a user find information on with a specific sub-goal in mind. "
        "You have been provided with a prompt to search for please provide a summary of the results. "
    )
)

In [ ]:
@orchestration_agent.tool
def search_tool(ctx: RunContext[Deps], sub_task: str) -> str:
    """Use search agent to find information on a sub-task"""
    
    search_dep = SearchDeps(search_client=ctx.deps.search_client)
    search_agent = ctx.deps.search_agent
    return search_agent.run(sub_task, deps=search_dep)


In [ ]:
@search_agent.tool
def search_tool(ctx: RunContext[SearchDeps], query: str) -> str:
    """Use search client to find information on a query"""
    return ctx.deps.search_client.search(query)

In [ ]:
result = orchestration_agent.run("Help me plan a trip to paris tomorrow", deps=Deps())

In [ ]:
message = await result

In [ ]:
message.data

_OrchestrationPlan(main_goal='Plan a trip to Paris tomorrow', sub_goals=['Determine the best mode of transportation to Paris', 'Identify key attractions to visit in Paris', 'Plan a rough itinerary for the day', 'Find accommodation options in Paris for the night'], plan=['Research transportation options to Paris and select the best one for departure tomorrow', 'Look up must-see tourist attractions in Paris and create a list', 'Based on the attractions, organize a timetable for a day visit', 'Search for hotels or short-term accommodations in Paris for the night of the visit'])

In [ ]:
_OrchestrationPlan.model_validate_json(message.new_messages()[1].parts[0].args.args_json)

_OrchestrationPlan(main_goal='Plan a trip to Paris tomorrow', sub_goals=['Determine the best mode of transportation to Paris', 'Identify key attractions to visit in Paris', 'Plan a rough itinerary for the day', 'Find accommodation options in Paris for the night'], plan=['Research transportation options to Paris and select the best one for departure tomorrow', 'Look up must-see tourist attractions in Paris and create a list', 'Based on the attractions, organize a timetable for a day visit', 'Search for hotels or short-term accommodations in Paris for the night of the visit'])

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()